In [2]:
# install the python client library for RabbitMQ
!pip install pika -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 8.5 MB/s eta 0:00:00


In [3]:
# RabbitMQ is not installed by default on a Colab VM, so this installs a broker
# directly inside the Colab session. this is only needed to have something to
# connect to for this demo - if you already have the dev RabbitMQ url/creds,
# skip this cell and just fill in the config cell below instead.
!apt-get install -y -qq rabbitmq-server > /dev/null

# start the rabbitmq service (this takes 5-10 seconds to come up)
!service rabbitmq-server start

[ OK ]


In [4]:
import time

# give the broker a few seconds to finish starting before we try to connect
time.sleep(8)

# check the broker actually came up
!rabbitmqctl status | head -5

Status of node rabbit@aa2a13812dfb ...
Runtime

OS PID: 2553
OS: Linux


In [5]:
'''
# ---- RabbitMQ connection config ----
# these are the dev RabbitMQ connection details. replace host/port/username/
# password/vhost with what was given for the dev environment once available.
# use_tls should be set to True if the dev broker requires an amqps connection.

RABBITMQ_HOST = 'localhost'      # dev RabbitMQ hostname goes here
RABBITMQ_PORT = 5672             # 5672 for plain amqp, 5671 for amqp over TLS
RABBITMQ_USERNAME = 'guest'      # dev username goes here
RABBITMQ_PASSWORD = 'guest'      # dev password goes here
RABBITMQ_VHOST = '/'             # dev vhost goes here
RABBITMQ_USE_TLS = False         # set True if the dev instance requires TLS
'''

RABBITMQ_HOST = '129.153.75.221'
RABBITMQ_PORT = 5672             # not given, using the standard amqp port since TLS is not required
RABBITMQ_USERNAME = 'bytesmart_interns'
RABBITMQ_PASSWORD = 'YaZU4ghFdBzY'
RABBITMQ_VHOST = '/'
RABBITMQ_USE_TLS = False

QUEUE_NAME = 'yield_prediction_queue'

In [6]:
# import the Python client library for RabbitMQ
import pika
# import the SSL module for secure communication and encrypted connections
import ssl

In [7]:
# builds a fresh connection to RabbitMQ using the config above.
# written as a function instead of one global connection object because pika
# connections are not thread safe, and this same function gets reused later
# in the phase 5 notebook where the app publishes from the flask thread and
# consumes from a separate background thread at the same time.
def get_rabbitmq_connection():

    credentials = pika.PlainCredentials(RABBITMQ_USERNAME, RABBITMQ_PASSWORD)

    if RABBITMQ_USE_TLS:
        # amqps connection, wrap the socket with ssl
        ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        ssl_options = pika.SSLOptions(ssl_context, RABBITMQ_HOST)
        connection_params = pika.ConnectionParameters(
            host=RABBITMQ_HOST,
            port=RABBITMQ_PORT,
            virtual_host=RABBITMQ_VHOST,
            credentials=credentials,
            ssl_options=ssl_options
        )
    else:
        connection_params = pika.ConnectionParameters(
            host=RABBITMQ_HOST,
            port=RABBITMQ_PORT,
            virtual_host=RABBITMQ_VHOST,
            credentials=credentials
        )

    return pika.BlockingConnection(connection_params)


# quick test that the connection actually works
test_connection = get_rabbitmq_connection()
print('connected to rabbitmq:', test_connection.is_open)
test_connection.close()


connected to rabbitmq: True


In [8]:
import json

# declare the queue we are going to publish to and consume from.
# durable=True so the queue survives a broker restart (messages sitting in it
# will also need to be published as persistent if we want them to survive too,
# see delivery_mode below).
connection = get_rabbitmq_connection()
channel = connection.channel()
channel.queue_declare(queue=QUEUE_NAME, durable=True)

print('queue declared:', QUEUE_NAME)

queue declared: yield_prediction_queue


In [9]:
# build a sample message. this represents an event our app might publish
# whenever a yield prediction gets made for an applicant.
sample_message = {
    'event': 'prediction_requested',
    'applicant_district': 'Madurai',
    'applicant_category': 'MBC',
    'requested_at': time.strftime('%Y-%m-%d %H:%M:%S')
}

In [10]:
# publish the message onto the queue as json text.
# delivery_mode=2 marks the message as persistent (written to disk, not just
# kept in memory) so it is not lost if the broker restarts before anyone
# consumes it.
channel.basic_publish(
    exchange='',
    routing_key=QUEUE_NAME,
    body=json.dumps(sample_message),
    properties=pika.BasicProperties(delivery_mode=2)
)

print('published message:')
print(sample_message)

published message:
{'event': 'prediction_requested', 'applicant_district': 'Madurai', 'applicant_category': 'MBC', 'requested_at': '2026-09-18 09:27:58'}


In [11]:
# consume the message back off the queue.
# basic_get pulls a single message and returns immediately (or returns None if
# the queue is empty) which is easier to work with in a notebook than
# start_consuming(), which blocks forever waiting for messages.

method_frame, header_frame, body = channel.basic_get(queue=QUEUE_NAME, auto_ack=False)

if method_frame:
    received_message = json.loads(body)
    print('received message:')
    print(received_message)
else:
    print('no message was waiting on the queue')

received message:
{'event': 'prediction_requested', 'applicant_district': 'Madurai', 'applicant_category': 'MBC', 'requested_at': '2026-09-18 09:27:58'}


In [12]:
# process the message that was received.
# a real consumer would do something useful here - update a record, trigger a
# notification, write to a database, etc. for this demo we just validate the
# fields we expect are present and print out what "processing" it looks like.

def process_message(message):
    required_keys = ['event', 'applicant_district', 'applicant_category', 'requested_at']
    missing_keys = [k for k in required_keys if k not in message]

    if missing_keys:
        print('message is missing keys, skipping:', missing_keys)
        return False

    print(f"processing '{message['event']}' event for an applicant from {message['applicant_district']} "
          f"({message['applicant_category']} category), requested at {message['requested_at']}")
    return True


processed_ok = process_message(received_message)

# only ack the message (tell rabbitmq it was handled) once processing actually
# succeeded. if processing had failed we would want the message to stay on
# the queue (or get requeued) rather than silently disappearing.
if processed_ok:
    channel.basic_ack(delivery_tag=method_frame.delivery_tag)
    print('message acked')

processing 'prediction_requested' event for an applicant from Madurai (MBC category), requested at 2026-09-18 09:27:58
message acked


In [13]:
# close the connection once we are done with it
connection.close()
print('connection closed')

connection closed
